# LeetCode 1094 — Car Pooling bằng Python

Link đề: https://leetcode.com/problems/car-pooling/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Heap (Priority Queue), Simulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        ...
```

Ghi chú: đáp án là **một giá trị `bool` duy nhất**. Xe chỉ đi về phía đông, mỗi chuyến `[numPassengers, from, to]` chiếm chỗ trên đoạn `[from, to)` (khách xuống xe **trước** khi khách mới lên ở cùng một vị trí). Điểm cốt lõi: tải của xe tại mỗi vị trí là **tổng chồng lấn các đoạn có trọng số**, tính nhanh bằng **mảng hiệu (difference array)** rồi tổng tích luỹ: `+p` tại `from`, `−p` tại `to`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: các chuyến đi có số hành khách và đoạn đường `[from, to)`; xe có sức chứa `capacity`; kiểm tra xe có đón trả được mọi chuyến không.
2. Nắm công cụ: mảng hiệu và tổng tích luỹ, quét sự kiện (sweep line) có thứ tự ưu tiên khi hoà, hàng đợi ưu tiên (min-heap) theo điểm trả khách.
3. Đi từ **mô phỏng từng km** → **mảng hiệu** → **quét sự kiện đã sắp xếp** → **min-heap** → **kiểm tra tại các điểm đón**.
4. Hiểu mấu chốt: tải lớn nhất đạt được tại một **điểm đón khách**; mọi cách đều tính `max_x Σ_{i: from_i ≤ x < to_i} p_i` rồi so với `capacity`.
5. Tránh lỗi kinh điển: coi `to` là vị trí còn có khách (lệch `1`), xử lý sai thứ tự khi vừa trả vừa đón cùng vị trí, so sánh `>=` thay vì `>`, chỉ cộng tổng mọi hành khách, mảng hiệu thiếu phần tử (`to = 1000`), nhầm hiệu với tổng tích luỹ.

## 2. Đề bài & ví dụ

Một chiếc xe có `capacity` chỗ trống, chỉ chạy **về phía đông** (không quay đầu). Cho số nguyên `capacity` và mảng `trips`, trong đó `trips[i] = [numPassengers_i, from_i, to_i]` nghĩa là chuyến thứ `i` có `numPassengers_i` hành khách, đón tại `from_i` và trả tại `to_i` (tính bằng km từ vị trí xuất phát của xe).

Hành khách được **trả trước** khi đón khách mới ở cùng một vị trí. Tại **mọi điểm** trên tuyến, tổng số hành khách trên xe **không được vượt quá** `capacity`.

Trả về `true` nếu có thể đón và trả mọi hành khách của mọi chuyến, ngược lại `false`.

**Ví dụ 1:** `trips = [[2,1,5],[3,3,7]]`, `capacity = 4` → `false` (tại km `3` có `2 + 3 = 5 > 4` hành khách).

**Ví dụ 2:** `trips = [[2,1,5],[3,3,7]]`, `capacity = 5` → `true` (tải lớn nhất `5`, không vượt `5`).

**Ràng buộc:** `1 <= trips.length <= 1000`; `1 <= numPassengers_i <= 100`; `0 <= from_i < to_i <= 1000`; `1 <= capacity <= 10^5`.

In [1]:
import heapq
import itertools
import random
import time
from typing import List

examples = [
    ([[2, 1, 5], [3, 3, 7]], 4, False),
    ([[2, 1, 5], [3, 3, 7]], 5, True),
]
for trips, cap, want in examples:
    print(f"trips = {trips}, capacity = {cap} -> {want}")

trips = [[2, 1, 5], [3, 3, 7]], capacity = 4 -> False
trips = [[2, 1, 5], [3, 3, 7]], capacity = 5 -> True


## 3. Kiến thức nền

### 3.1 Đoạn nửa mở `[from, to)`

Hành khách của chuyến `i` ở trên xe tại các vị trí `x` với `from_i ≤ x < to_i`. Tại `x = to_i` họ **đã xuống** (đề nói trả trước khi đón ở cùng vị trí). Do đó tải tại `x` là

`load(x) = Σ_{i : from_i ≤ x < to_i} numPassengers_i`.

Xe thoả đề ⇔ `load(x) ≤ capacity` với mọi `x`.

In [2]:
def load_at(trips, x):
    """Tải của xe tại km x theo định nghĩa (đoạn nửa mở [from, to))."""
    return sum(p for p, a, b in trips if a <= x < b)


print([load_at(examples[0][0], x) for x in range(9)])
assert [load_at(examples[0][0], x) for x in range(9)] == [0, 2, 2, 5, 5, 3, 3, 0, 0]

[0, 2, 2, 5, 5, 3, 3, 0, 0]


### 3.2 Mảng hiệu (difference array)

Muốn "cộng `p` cho mọi `x ∈ [a, b)`" nhanh: `diff[a] += p`, `diff[b] -= p`; khi đó `load(x) = diff[0] + … + diff[x]` (tổng tích luỹ). Mỗi chuyến chỉ tốn `O(1)` cập nhật, rồi một lần quét cho toàn tuyến.

### 3.3 Tải lớn nhất đạt tại điểm đón

`load(x)` chỉ **tăng** tại các điểm `x = from_i` (đón thêm) và chỉ giảm tại các điểm trả khách. Do đó giá trị lớn nhất của `load` đạt tại một điểm đón (hoặc bằng `0` nếu không có gì). Điều này cho phép chỉ kiểm tra `n` điểm `from_i`.

## 4. Ý tưởng nền tảng / trực giác

Hãy nghĩ xe đi từ km `0` đến km `1000`: mỗi lần tới một điểm có người đón thì số khách tăng, có người xuống thì giảm. Ta chỉ cần **biết số khách tại mỗi điểm** và kiểm tra có quá `capacity` không. Thay vì cộng từng chuyến cho từng km (`O(n · 1000)`), ghi nhận **thay đổi** tại điểm đầu và điểm cuối của mỗi chuyến rồi cộng dồn.

## 5. Brute force — mô phỏng từng km theo định nghĩa

**Ý tưởng:** với mỗi `x = 0..1000`, cộng số khách của các chuyến thoả `from ≤ x < to`; nếu vượt `capacity` thì trả `False`. Đúng theo định nghĩa, làm **oracle** và chạy được vì khoảng `0..1000` nhỏ (`≈ 10^6` phép tính).

In [3]:
def car_pool_brute(trips, capacity):
    for x in range(1001):
        if load_at(trips, x) > capacity:
            return False
    return True


for trips, cap, want in examples:
    assert car_pool_brute(trips, cap) is want
print([car_pool_brute(t, c) for t, c, _ in examples])

[False, True]


In [4]:
# Code nộp LeetCode cho Brute force (O(1001 · n) ≈ 10^6: vẫn qua được vì ràng buộc nhỏ)
class SolutionBrute:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        for x in range(1001):
            load = 0
            for p, a, b in trips:
                if a <= x < b:
                    load += p
            if load > capacity:
                return False
        return True

**Độ phức tạp:** `O(L · n)` với `L = 1001` vị trí, `n` chuyến; `O(1)` bộ nhớ.

**Nút thắt:** mỗi chuyến được cộng lại ở mọi km nó đi qua — nên chỉ ghi nhận **thay đổi** ở hai đầu.

## 6. Bước đột phá

Dùng mảng hiệu: `diff[from] += p`, `diff[to] -= p`; quét `x = 0..1000` cộng dồn `cur += diff[x]`, kiểm tra `cur ≤ capacity`. `O(n + L)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** mảng hiệu (đáp án chính), **Cách 3** quét sự kiện đã sắp xếp, **Cách 4** min-heap theo điểm trả, **Cách 5** kiểm tra tại các điểm đón.

### Cách 2 — Mảng hiệu (đáp án chính)

**Giải thích:** `diff` độ dài `1002` (chỉ số `0..1001` an toàn cho `to ≤ 1000`); `diff[a] += p`, `diff[b] -= p`; quét cộng dồn.

In [5]:
def car_pool_diff(trips, capacity):
    diff = [0] * 1002
    for p, a, b in trips:
        diff[a] += p                                      # khách lên tại a
        diff[b] -= p                                      # khách xuống tại b (trước khi đón ở b)
    cur = 0
    for x in range(1001):
        cur += diff[x]                                    # cur = tải tại km x
        if cur > capacity:
            return False
    return True


for trips, cap, want in examples:
    assert car_pool_diff(trips, cap) is want
print([car_pool_diff(t, c) for t, c, _ in examples])

[False, True]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionDiff:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        diff = [0] * 1002
        for p, a, b in trips:
            diff[a] += p
            diff[b] -= p
        cur = 0
        for d in diff:
            cur += d
            if cur > capacity:
                return False
        return True

**Độ phức tạp:** `O(n + L)` thời gian, `O(L)` bộ nhớ (`L = 1002`).

### Cách 3 — Quét sự kiện đã sắp xếp

**Ý tưởng:** mỗi chuyến sinh hai sự kiện `(a, +p)` và `(b, −p)`. Sắp xếp theo `(vị trí, delta)`: với cùng vị trí, `delta` **âm đứng trước** (`−p < +p`) nên **trả trước khi đón** — đúng điều đề yêu cầu. Quét cộng dồn.

In [7]:
def car_pool_events(trips, capacity):
    events = []
    for p, a, b in trips:
        events.append((a, p))
        events.append((b, -p))
    events.sort()                                         # cùng vị trí: delta âm (trả) đứng trước delta dương (đón)
    cur = 0
    for pos, delta in events:
        cur += delta
        if cur > capacity:
            return False
    return True


for trips, cap, want in examples:
    assert car_pool_events(trips, cap) is want
print([car_pool_events(t, c) for t, c, _ in examples])

[False, True]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionEvents:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        events = []
        for p, a, b in trips:
            events.append((a, p))
            events.append((b, -p))
        events.sort()
        cur = 0
        for _, delta in events:
            cur += delta
            if cur > capacity:
                return False
        return True

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Không phụ thuộc phạm vi toạ độ nên dùng được khi vị trí rất lớn.

### Cách 4 — Min-heap theo điểm trả

**Ý tưởng:** sắp xếp chuyến theo điểm đón. Giữ min-heap `(to, p)` các chuyến đang trên xe và tổng `cur`. Khi tới chuyến có điểm đón `a`, **pop** mọi chuyến có `to ≤ a` (đã xuống), rồi thêm chuyến mới và kiểm tra `cur ≤ capacity`.

In [9]:
def car_pool_heap(trips, capacity):
    cur = 0
    heap = []                                             # (to, p) các chuyến đang trên xe
    for p, a, b in sorted(trips, key=lambda t: t[1]):
        while heap and heap[0][0] <= a:                   # xuống xe trước khi đón (to <= a)
            cur -= heapq.heappop(heap)[1]
        cur += p
        heapq.heappush(heap, (b, p))
        if cur > capacity:
            return False
    return True


for trips, cap, want in examples:
    assert car_pool_heap(trips, cap) is want
print([car_pool_heap(t, c) for t, c, _ in examples])

[False, True]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionHeap:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        cur = 0
        heap = []
        for p, a, b in sorted(trips, key=lambda t: t[1]):
            while heap and heap[0][0] <= a:
                cur -= heapq.heappop(heap)[1]
            cur += p
            heapq.heappush(heap, (b, p))
            if cur > capacity:
                return False
        return True

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — Chỉ kiểm tra tại các điểm đón

**Ý tưởng:** theo mục 3.3, tải lớn nhất đạt tại một điểm đón. Với mỗi chuyến `i`, tính `load(from_i)` bằng định nghĩa (cộng các chuyến `j` có `from_j ≤ from_i < to_j`) và so với `capacity`. `O(n²)` nhưng đơn giản và không cần mảng/sắp xếp.

In [11]:
def car_pool_pickups(trips, capacity):
    for _, a, _ in trips:
        if load_at(trips, a) > capacity:
            return False
    return True


for trips, cap, want in examples:
    assert car_pool_pickups(trips, cap) is want
print([car_pool_pickups(t, c) for t, c, _ in examples])

[False, True]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionPickups:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        for _, a, _ in trips:
            load = 0
            for p, a2, b2 in trips:
                if a2 <= a < b2:
                    load += p
            if load > capacity:
                return False
        return True

**Độ phức tạp:** `O(n²)` thời gian (`10^6` với `n = 1000`), `O(1)` bộ nhớ phụ.

## 8. TRACE từng bước

Trace **Cách 2** với `trips = [[2,1,5],[3,3,7]]`, `capacity = 4`:

`diff[1] += 2`, `diff[5] -= 2`, `diff[3] += 3`, `diff[7] -= 3`. Quét:

| `x` | `diff[x]` | `cur` (tải tại km `x`) | so với `capacity = 4` |
|---|---|---|---|
| 0 | 0 | 0 | ok |
| 1 | +2 | 2 | ok |
| 2 | 0 | 2 | ok |
| 3 | +3 | 5 | **5 > 4: trả `False`** |

Với `capacity = 5`: tiếp tục `x = 4: 5`, `x = 5: 3`, `x = 6: 3`, `x = 7: 0` ⇒ tải lớn nhất `5 ≤ 5` ⇒ `True`. Hàm trace in từng bước:

In [13]:
def trace_diff(trips, capacity):
    diff = {}
    for p, a, b in trips:
        diff[a] = diff.get(a, 0) + p
        diff[b] = diff.get(b, 0) - p
    print(f"trips = {trips}, capacity = {capacity}, diff = {dict(sorted(diff.items()))}")
    cur = 0
    for x in sorted(diff):
        cur += diff[x]
        status = "ok" if cur <= capacity else f"vượt {capacity}: trả False"
        print(f"  km {x}: diff {diff[x]:+d} -> tải {cur} ({status})")
        if cur > capacity:
            return False
    return True


assert trace_diff([[2, 1, 5], [3, 3, 7]], 4) is False
assert trace_diff([[2, 1, 5], [3, 3, 7]], 5) is True

trips = [[2, 1, 5], [3, 3, 7]], capacity = 4, diff = {1: 2, 3: 3, 5: -2, 7: -3}
  km 1: diff +2 -> tải 2 (ok)
  km 3: diff +3 -> tải 5 (vượt 4: trả False)
trips = [[2, 1, 5], [3, 3, 7]], capacity = 5, diff = {1: 2, 3: 3, 5: -2, 7: -3}
  km 1: diff +2 -> tải 2 (ok)
  km 3: diff +3 -> tải 5 (ok)
  km 5: diff -2 -> tải 3 (ok)
  km 7: diff -3 -> tải 0 (ok)


**Trace thứ tự hoà (Cách 3):** `trips = [[3,1,5],[3,5,9]]`, `capacity = 3`. Sự kiện: `(1,+3)`, `(5,-3)`, `(5,+3)`, `(9,-3)`. Sau sắp xếp: `(1,+3), (5,-3), (5,+3), (9,-3)` (cùng vị trí `5`, `-3` đứng trước `+3`): tải `3, 0, 3, 0` ⇒ `True`. Nếu sắp xếp sai (đón trước khi trả) thì tải tạm `6 > 3` ⇒ sai.

In [14]:
assert car_pool_events([[3, 1, 5], [3, 5, 9]], 3) is True
assert sorted([(1, 3), (5, -3), (5, 3), (9, -3)]) == [(1, 3), (5, -3), (5, 3), (9, -3)]
print("trả trước đón: True")

trả trước đón: True


## 9. Code nộp LeetCode

Chọn **Cách 2** (mảng hiệu). Giữ nguyên `class Solution`, `carPooling(self, trips, capacity)`:

In [15]:
class Solution:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        diff = [0] * 1002                           # to <= 1000 nên chỉ số tối đa 1000; dư một ô cho an toàn
        for p, a, b in trips:
            diff[a] += p                            # khách lên tại km a
            diff[b] -= p                            # khách xuống tại km b (trước khi đón ở b)
        cur = 0
        for d in diff:
            cur += d                                # tổng tích luỹ = số khách trên xe tại km này
            if cur > capacity:
                return False
        return True


print([Solution().carPooling(t, c) for t, c, _ in examples])
assert all(Solution().carPooling(t, c) is w for t, c, w in examples)

[False, True]


## 10. Vì sao đúng

**(a) Mảng hiệu cho đúng tải.** Với mỗi chuyến `(p, a, b)`, đóng góp vào `diff` là `+p` tại `a` và `-p` tại `b`. Tổng tích luỹ `cur(x) = Σ_{y ≤ x} diff[y]` đóng góp của chuyến này là `p` nếu `a ≤ x` và `b > x` (tức `a ≤ x < b`) và `0` ngược lại (nếu `x < a`: chưa có gì; nếu `x ≥ b`: `+p - p = 0`). Cộng mọi chuyến: `cur(x) = Σ_{i: a_i ≤ x < b_i} p_i = load(x)`.

**(b) Đúng với luật "trả trước khi đón".** Tại `x = b_i`, chuyến `i` đã đóng góp `0` (vì `x ≥ b_i`), còn chuyến mới đón tại `x` đóng góp `p` (vì `a ≤ x`): tải tại `x` đã **loại** khách vừa xuống và **gồm** khách vừa lên, đúng với định nghĩa đề. Bản `x = a_i = b_j` không bao giờ tính cả hai chuyến cùng lúc.

**(c) Điều kiện đúng.** Xe thoả đề ⇔ `load(x) ≤ capacity` với mọi `x ∈ [0, 1000]`; vòng lặp kiểm tra đúng điều đó và dừng ngay khi vi phạm (so sánh `>`: bằng `capacity` vẫn hợp lệ).

**(d) Quét sự kiện (Cách 3).** Sắp xếp theo `(vị trí, delta)`: khi hoà vị trí, `delta` âm trước; giá trị `cur` sau mỗi sự kiện là tải **tạm thời** ở trạng thái "đã xử lý các sự kiện đến thời điểm đó". Vì mọi trả khách ở vị trí `x` được xử lý trước mọi đón ở `x`, các giá trị trung gian `cur` không bao giờ lớn hơn `load(x)` (không có trạng thái "vừa đón vừa chưa trả"), và giá trị cuối của nhóm sự kiện tại `x` đúng bằng `load(x)`; do đó max của `cur` bằng max của `load`.

**(e) Min-heap (Cách 4).** Xử lý chuyến theo điểm đón tăng dần. Trước khi thêm chuyến có điểm đón `a`, loại mọi chuyến có `to ≤ a` (đã xuống). Khi đó `cur` bằng tải ngay sau khi đón tại `a`, chính là `load(a)` đối với các chuyến có điểm đón `≤ a` (các chuyến có điểm đón **bằng** `a` được thêm lần lượt nên `cur` tăng dần tới `load(a)`; kiểm tra tại mỗi lần thêm, giá trị cuối cùng lớn nhất). Tải lớn nhất đạt tại điểm đón (mục 3.3) nên kiểm tra tại các điểm đón là đủ.

**(f) Cách 5.** Theo (3.3): `max_x load(x) = max_i load(a_i)`; kiểm tra `load(a_i) ≤ capacity` với mọi `i` tương đương `max load ≤ capacity`.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle mô phỏng từng km trên mọi danh sách nhỏ và hàng nghìn danh sách ngẫu nhiên tới `1000` chuyến (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng km | `O(L · n)` | `O(1)` | oracle, `L = 1001` |
| 2. Mảng hiệu | `O(n + L)` | `O(L)` | **đáp án chính** |
| 3. Quét sự kiện | `O(n log n)` | `O(n)` | không phụ thuộc phạm vi toạ độ |
| 4. Min-heap | `O(n log n)` | `O(n)` | quen thuộc từ Meeting Rooms II |
| 5. Điểm đón | `O(n²)` | `O(1)` | đơn giản, `n ≤ 1000` |

Đo thực tế (`n = 1000` chuyến):

In [16]:
def rand_trips(rng, n, maxpos=1000, maxp=100):
    out = []
    for _ in range(n):
        a = rng.randint(0, maxpos - 1)
        b = rng.randint(a + 1, maxpos)
        out.append([rng.randint(1, maxp), a, b])
    return out


def bench(fn, trips, cap, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(trips, cap)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1094)
trips = rand_trips(rng, 1000)
cap = 10 ** 5                                            # đủ lớn để các cách phải quét hết
for name, fn in [("brute", car_pool_brute), ("diff", car_pool_diff), ("events", car_pool_events), ("heap", car_pool_heap), ("pickups", car_pool_pickups)]:
    print(f"{name:8s}: {bench(fn, trips, cap) * 1000:8.2f} ms -> {fn(trips, cap)}")

brute   :    47.78 ms -> True
diff    :     0.13 ms -> True
events  :     0.56 ms -> True
heap    :     0.56 ms -> True
pickups :    48.45 ms -> True


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi danh sách nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "diff": car_pool_diff,
    "events": car_pool_events,
    "heap": car_pool_heap,
    "pickups": car_pool_pickups,
    "Solution": lambda t, c: Solution().carPooling([r[:] for r in t], c),
    "SolutionDiff": lambda t, c: SolutionDiff().carPooling([r[:] for r in t], c),
    "SolutionEvents": lambda t, c: SolutionEvents().carPooling([r[:] for r in t], c),
    "SolutionHeap": lambda t, c: SolutionHeap().carPooling([r[:] for r in t], c),
    "SolutionPickups": lambda t, c: SolutionPickups().carPooling([r[:] for r in t], c),
}
BRUTE = {"brute": car_pool_brute, "SolutionBrute": lambda t, c: SolutionBrute().carPooling([r[:] for r in t], c)}


def max_load(trips):
    return max(load_at(trips, x) for x in range(1001))


def check(trips, cap, want=None, brute=True):
    if want is None:
        want = max_load(trips) <= cap if brute else Solution().carPooling(trips, cap)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(trips, cap)
        assert got is want, (name, trips, cap, got, want)
    return want


for t, c, w in examples:
    check(t, c, w)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- Một chuyến, `capacity` đúng bằng số khách (hợp lệ: `≤`) và `capacity` ít hơn `1`.
- Trả và đón cùng vị trí (`to_i = from_j`): không tính chồng.
- Hai chuyến trùng hoàn toàn.
- Chuyến bắt đầu `0` và kết thúc `1000` (biên).
- Nhiều chuyến cùng bắt đầu/kết thúc tại một điểm.
- Chuyến lồng nhau, chuyến nối tiếp không chồng.
- `capacity = 1` với chuyến `1` hành khách chồng lấn / không chồng lấn.

In [18]:
edge = [
    ([[5, 0, 1000]], 5, True), ([[5, 0, 1000]], 4, False),
    ([[3, 1, 5], [3, 5, 9]], 3, True), ([[3, 1, 5], [3, 5, 9]], 2, False),
    ([[2, 1, 5], [2, 1, 5]], 4, True), ([[2, 1, 5], [2, 1, 5]], 3, False),
    ([[1, 0, 1], [1, 0, 1], [1, 0, 1]], 3, True), ([[1, 0, 1], [1, 0, 1], [1, 0, 1]], 2, False),
    ([[2, 0, 10], [3, 2, 8], [4, 4, 6]], 9, True), ([[2, 0, 10], [3, 2, 8], [4, 4, 6]], 8, False),
    ([[1, 0, 1], [1, 1, 2], [1, 2, 3]], 1, True), ([[1, 0, 2], [1, 1, 3]], 1, False),
    ([[100, 999, 1000], [100, 0, 1]], 100, True), ([[100] * 1 + [999, 1000], [1, 999, 1000]], 100, False),
]
for t, c, w in edge:
    got = check(t, c, w)
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

14 edge case: mọi cài đặt (kể cả vét cạn) khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_pickup_before_drop(trips, capacity):            # cùng vị trí: đón trước khi trả (sắp xếp delta giảm dần khi hoà)
    events = []
    for p, a, b in trips:
        events.append((a, -p))                          # dùng -p để đón (delta dương) đứng trước khi sắp xếp
        events.append((b, p))
    events.sort()
    cur = 0
    for _, key in events:
        cur += -key
        if cur > capacity:
            return False
    return True


def bug_inclusive_end(trips, capacity):                 # coi km `to` vẫn còn khách (diff[to + 1])
    diff = [0] * 1003
    for p, a, b in trips:
        diff[a] += p
        diff[b + 1] -= p
    cur = 0
    for d in diff:
        cur += d
        if cur > capacity:
            return False
    return True


def bug_total_passengers(trips, capacity):              # tổng mọi hành khách <= capacity
    return sum(p for p, _, _ in trips) <= capacity


def bug_strict_capacity(trips, capacity):               # đòi tải < capacity (bằng cũng không được)
    diff = [0] * 1002
    for p, a, b in trips:
        diff[a] += p
        diff[b] -= p
    cur = 0
    for d in diff:
        cur += d
        if cur >= capacity:
            return False
    return True


def bug_check_diff_not_prefix(trips, capacity):         # kiểm tra từng diff thay vì tổng tích luỹ
    diff = [0] * 1002
    for p, a, b in trips:
        diff[a] += p
        diff[b] -= p
    return all(d <= capacity for d in diff)


def bug_small_array(trips, capacity):                   # mảng hiệu thiếu một ô: to = 1000 gây IndexError
    diff = [0] * 1000
    for p, a, b in trips:
        diff[a] += p
        diff[b] -= p
    cur = 0
    for d in diff:
        cur += d
        if cur > capacity:
            return False
    return True


def bug_heap_no_pop_equal(trips, capacity):             # pop khi to < a (thiếu dấu bằng): không trả trước khi đón
    cur = 0
    heap = []
    for p, a, b in sorted(trips, key=lambda t: t[1]):
        while heap and heap[0][0] < a:
            cur -= heapq.heappop(heap)[1]
        cur += p
        heapq.heappush(heap, (b, p))
        if cur > capacity:
            return False
    return True


BUGS = [bug_pickup_before_drop, bug_inclusive_end, bug_total_passengers, bug_strict_capacity, bug_check_diff_not_prefix, bug_small_array, bug_heap_no_pop_equal]
rng = random.Random(10940)
pool = []
for _ in range(3000):
    n = rng.randint(1, 5)
    trips = []
    for _ in range(n):
        a = rng.choice([0, 1, 2, 3, 998, 999])
        b = rng.choice([x for x in (1, 2, 3, 4, 5, 999, 1000) if x > a])
        trips.append([rng.randint(1, 3), a, b])
    pool.append((trips, rng.randint(1, 8)))
for bug in BUGS:
    found = None
    for t, c in pool:
        try:
            got = bug(t, c)
        except IndexError:
            got = "IndexError"
        if got != (max_load(t) <= c):
            found = (t, c, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên trips = {found[0]}, capacity = {found[1]}: ra {found[2]}, đúng {max_load(found[0]) <= found[1]}")

bug_pickup_before_drop     sai trên trips = [[1, 0, 3], [2, 3, 999]], capacity = 2: ra False, đúng True
bug_inclusive_end          sai trên trips = [[1, 0, 3], [2, 3, 999]], capacity = 2: ra False, đúng True
bug_total_passengers       sai trên trips = [[1, 1, 3], [1, 0, 2], [2, 0, 3], [3, 999, 1000]], capacity = 4: ra False, đúng True
bug_strict_capacity        sai trên trips = [[2, 998, 1000]], capacity = 2: ra False, đúng True
bug_check_diff_not_prefix  sai trên trips = [[1, 1, 999], [3, 3, 5], [3, 998, 1000], [1, 2, 999], [3, 3, 4]], capacity = 7: ra True, đúng False
bug_small_array            sai trên trips = [[2, 998, 1000]], capacity = 2: ra IndexError, đúng True
bug_heap_no_pop_equal      sai trên trips = [[1, 0, 3], [2, 3, 999]], capacity = 2: ra False, đúng True


### D. Vét cạn mọi danh sách nhỏ

Mọi danh sách `1..3` chuyến, mỗi chuyến có `numPassengers ∈ {1, 2}` và `(from, to)` với `0 ≤ from < to ≤ 3` (`6` đoạn): `12` chuyến khả dĩ, `12 + 144 + 1728 = 1 884` danh sách (thứ tự có nghĩa), nhân với `capacity = 1..6`. So mọi cài đặt với oracle mô phỏng từng km.

In [20]:
t0 = time.perf_counter()
cnt = 0
unit = [[p, a, b] for p in (1, 2) for a in range(4) for b in range(a + 1, 4)]
for n in (1, 2, 3):
    for combo in itertools.product(unit, repeat=n):
        trips = [list(x) for x in combo]
        ml = max_load(trips)
        for cap in range(1, 7):
            want = ml <= cap
            for name, fn in IMPLS.items():
                assert fn(trips, cap) is want, (name, trips, cap)
            cnt += 1
print(cnt, "cặp (danh sách, capacity):", len(IMPLS), "cài đặt khớp oracle;", round(time.perf_counter() - t0, 1), "s")

11304 cặp (danh sách, capacity): 9 cài đặt khớp oracle; 1.4 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Danh sách ngẫu nhiên nhỏ (`n ≤ 30`, vị trí tới `1000`): mọi cách khớp oracle; (2) `n` lớn: các cách khớp nhau; (3) **kết quả bằng `max_load ≤ capacity`** và **đơn điệu theo `capacity`**; (4) **hoán vị chuyến** không đổi; (5) **dịch vị trí** (cộng hằng, giữ trong `[0, 1000]`) không đổi; (6) **tách một chuyến** `(p, a, b)` thành `(p, a, m)` và `(p, m, b)` không đổi (vì trả trước đón tại `m`); (7) **thêm chuyến** không đổi `False → True`; (8) **nhân số khách và capacity** với `k` không đổi.

In [21]:
rng = random.Random(20240194)
t0 = time.perf_counter()
for it in range(600):
    n = rng.randint(1, 30)
    trips = rand_trips(rng, n, maxpos=rng.choice([10, 50, 1000]), maxp=rng.choice([3, 10, 100]))
    ml = max_load(trips)
    for cap in {max(1, ml - 1), ml, ml + 1, rng.randint(1, 2 * ml + 1)}:
        want = ml <= cap
        check(trips, cap, want)                                                       # (1), (3)
    cap = max(1, ml)
    assert Solution().carPooling(trips, cap) is True and (ml <= 1 or Solution().carPooling(trips, ml - 1) is False)
    shuffled = [r[:] for r in trips]
    rng.shuffle(shuffled)
    assert Solution().carPooling(shuffled, ml - 1 if ml > 1 else ml) is (ml - 1 >= ml if ml > 1 else True)   # (4)
    maxp = max(b for _, _, b in trips)
    minp = min(a for _, a, _ in trips)
    delta = rng.randint(0, 1000 - maxp)
    moved = [[p, a + delta, b + delta] for p, a, b in trips]
    assert Solution().carPooling(moved, cap) is True and (ml <= 1 or Solution().carPooling(moved, ml - 1) is False)   # (5)
    i = rng.randrange(n)
    p, a, b = trips[i]
    if b - a >= 2:
        m = rng.randint(a + 1, b - 1)
        split = trips[:i] + [[p, a, m], [p, m, b]] + trips[i + 1:]
        assert max_load(split) == ml                                                   # (6)
    extra = rand_trips(rng, 1)
    if ml > 1:
        assert Solution().carPooling(trips + extra, ml - 1) is False                  # (7)
    k = rng.randint(2, 4)
    scaled = [[p * k, a, b] for p, a, b in trips]
    assert Solution().carPooling(scaled, ml * k) is True and Solution().carPooling(scaled, ml * k - 1) is False   # (8)
print("(1)-(8) 600 danh sách ngẫu nhiên:", round(time.perf_counter() - t0, 1), "s")

(1)-(8) 600 danh sách ngẫu nhiên: 4.1 s


### F. Giới hạn

`n = 1000`: chuyến `[100, 0, 1000]` chồng lấn tất cả (tải `10^5`, `capacity = 10^5` hợp lệ, `10^5 - 1` không); chuyến nối tiếp (`[100, i, i+1]`); chuyến ngẫu nhiên; hai nhóm sát biên. Các cách nhanh khớp nhau.

In [22]:
t0 = time.perf_counter()
cases = [
    ([[100, 0, 1000]] * 1000, 10 ** 5, True),
    ([[100, 0, 1000]] * 1000, 10 ** 5 - 1, False),
    ([[100, i, i + 1] for i in range(1000)], 100, True),
    ([[100, i, i + 1] for i in range(1000)], 99, False),
    ([[1, 0, 1000] for _ in range(1000)], 1000, True),
]
rg = random.Random(21)
for _ in range(6):
    trips = rand_trips(rg, 1000)
    ml = max_load(trips)
    cases.append((trips, ml, True))
    cases.append((trips, ml - 1, False))
for trips, cap, want in cases:
    got = check(trips, cap, want, brute=False)
    assert got is (max_load(trips) <= cap)
print(len(cases), "test giới hạn n = 1000:", round(time.perf_counter() - t0, 1), "s")

17 test giới hạn n = 1000: 2.5 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đón trước khi trả ở cùng vị trí

In [23]:
t, c = [[3, 1, 5], [3, 5, 9]], 3
print("bug:", bug_pickup_before_drop(t, c), "| đúng:", Solution().carPooling(t, c))
assert bug_pickup_before_drop(t, c) is False and Solution().carPooling(t, c) is True

bug: False | đúng: True


**Vì sao sai:** đề nói khách xuống trước khi khách mới lên cùng vị trí. Xử lý đón trước làm tải tạm thời `6 > 3` dù thực tế chỉ `3`.

### Lỗi 2: coi km `to` vẫn còn khách

In [24]:
print("bug:", bug_inclusive_end(t, c), "| đúng:", Solution().carPooling(t, c))
assert bug_inclusive_end(t, c) is False

bug: False | đúng: True


**Vì sao sai:** `diff[b + 1] -= p` coi khách vẫn có mặt tại `b`; đoạn là **nửa mở** `[a, b)`. Phải `diff[b] -= p`.

### Lỗi 3: chỉ cộng tổng mọi hành khách

In [25]:
t, c = [[1, 0, 1], [1, 1, 2], [1, 2, 3]], 1
print("bug:", bug_total_passengers(t, c), "| đúng:", Solution().carPooling(t, c))
assert bug_total_passengers(t, c) is False and Solution().carPooling(t, c) is True

bug: False | đúng: True


**Vì sao sai:** các chuyến không chồng lấn không cùng lúc trên xe; chỉ tổng hành khách **cùng một thời điểm** mới ràng buộc.

### Lỗi 4: so sánh `>=` thay vì `>`

In [26]:
t, c = [[5, 0, 10]], 5
print("bug:", bug_strict_capacity(t, c), "| đúng:", Solution().carPooling(t, c))
assert bug_strict_capacity(t, c) is False and Solution().carPooling(t, c) is True

bug: False | đúng: True


**Vì sao sai:** tải **bằng** `capacity` vẫn hợp lệ (ví dụ 2 của đề). Chỉ `> capacity` là vi phạm.

### Lỗi 5: kiểm tra từng phần tử mảng hiệu thay vì tổng tích luỹ

In [27]:
t, c = [[3, 0, 5], [3, 2, 6]], 4
print("bug:", bug_check_diff_not_prefix(t, c), "| đúng:", Solution().carPooling(t, c))
assert bug_check_diff_not_prefix(t, c) is True and Solution().carPooling(t, c) is False

bug: True | đúng: False


**Vì sao sai:** `diff[x]` là **thay đổi** tại `x`, không phải tải. Tải là tổng tích luỹ `diff[0] + … + diff[x]`.

### Lỗi 6 (bonus): mảng hiệu thiếu ô, `to = 1000`

In [28]:
t, c = [[1, 0, 1000]], 5
try:
    bug_small_array(t, c)
    print("không lỗi")
except IndexError as e:
    print("IndexError:", e)
assert Solution().carPooling(t, c) is True

IndexError: list index out of range


**Vì sao sai:** `to` có thể bằng `1000` nên mảng hiệu cần **ít nhất `1001`** phần tử (chỉ số `0..1000`). Dùng `[0] * 1002` (hoặc `max(to) + 2`).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "nhiều đoạn có trọng số, hỏi tải/độ phủ lớn nhất tại một điểm, so với giới hạn" → **mảng hiệu** (toạ độ nhỏ) hoặc **quét sự kiện** (toạ độ lớn) hoặc **min-heap**.

**Mẹo:**

1. Đoạn nửa mở `[a, b)`: `+p` tại `a`, `-p` tại `b`.
2. Sắp xếp sự kiện theo `(vị trí, delta)` để trả trước đón khi hoà.
3. Tải lớn nhất đạt tại một điểm đón: kiểm tra `n` điểm là đủ.
4. Kích thước mảng hiệu `max(to) + 2` để không tràn.
5. Oracle: mô phỏng từng km theo định nghĩa (`load_at`).
6. Quen thuộc: Meeting Rooms II, My Calendar III, Corporate Flight Bookings.

**Họ bài cùng pattern:** Meeting Rooms II (253), My Calendar III (732), Corporate Flight Bookings (1109), Range Addition (370), Maximum Population Year (1854), Minimum Number of Arrows to Burst Balloons (452).

## 15. Biến thể

1. **Sức chứa tối thiểu** cần để thực hiện mọi chuyến: `max_x load(x)`.
2. **Km đầu tiên** đạt tải lớn nhất.
3. **Tổng chiều dài** các đoạn có tải vượt `capacity`.
4. **Số phòng họp tối thiểu** (Meeting Rooms II): `capacity` là số phòng, mỗi cuộc họp `p = 1`.

Cài đặt và đối chiếu mô phỏng từng km:

In [29]:
def min_capacity(trips):
    diff = {}
    for p, a, b in trips:
        diff[a] = diff.get(a, 0) + p
        diff[b] = diff.get(b, 0) - p
    cur = best = 0
    for x in sorted(diff):
        cur += diff[x]
        best = max(best, cur)
    return best


def first_peak(trips):
    diff = {}
    for p, a, b in trips:
        diff[a] = diff.get(a, 0) + p
        diff[b] = diff.get(b, 0) - p
    cur = best = 0
    pos = None
    for x in sorted(diff):
        cur += diff[x]
        if cur > best:
            best, pos = cur, x
    return pos, best


def length_over(trips, capacity):
    diff = {}
    for p, a, b in trips:
        diff[a] = diff.get(a, 0) + p
        diff[b] = diff.get(b, 0) - p
    cur, total, prev = 0, 0, None
    for x in sorted(diff):
        if prev is not None and cur > capacity:
            total += x - prev                           # tải cur kéo dài trên [prev, x)
        cur += diff[x]
        prev = x
    return total


def meeting_rooms(intervals):
    return min_capacity([[1, s, e] for s, e in intervals])


rng = random.Random(15)
for _ in range(400):
    trips = rand_trips(rng, rng.randint(1, 12), maxpos=rng.choice([8, 30, 1000]), maxp=5)
    loads = [load_at(trips, x) for x in range(1001)]
    assert min_capacity(trips) == max(loads)
    pos, best = first_peak(trips)
    assert best == max(loads) and pos == loads.index(max(loads))
    cap = rng.randint(0, max(loads))
    assert length_over(trips, cap) == sum(1 for v in loads if v > cap)
print("sức chứa tối thiểu / km đỉnh đầu tiên / độ dài vượt capacity: khớp mô phỏng từng km")
assert meeting_rooms([(0, 30), (5, 10), (15, 20)]) == 2 and meeting_rooms([(7, 10), (2, 4)]) == 1

sức chứa tối thiểu / km đỉnh đầu tiên / độ dài vượt capacity: khớp mô phỏng từng km


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng km | định nghĩa | `O(L n)` | `O(1)` | oracle |
| 2. Mảng hiệu | `+p` / `-p` + tích luỹ | `O(n + L)` | `O(L)` | **đáp án chính** (toạ độ nhỏ) |
| 3. Quét sự kiện | sắp xếp `(pos, delta)` | `O(n log n)` | `O(n)` | toạ độ lớn |
| 4. Min-heap | pop khi `to <= from` | `O(n log n)` | `O(n)` | cách nhìn "đang trên xe" |
| 5. Tại điểm đón | `load(from_i)` | `O(n²)` | `O(1)` | đơn giản |

Mọi cách khớp oracle trên toàn bộ danh sách nhỏ (`≤ 3` chuyến) và `600` danh sách ngẫu nhiên.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().carPooling([[1, 0, 5], [1, 5, 10]], 1), Solution().carPooling([[2, 1, 3], [2, 2, 4]], 3), Solution().carPooling([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 8), Solution().carPooling([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 9))

True False False True


### Bài 2 (tính tay)

`trips = [[2,0,4],[3,1,3],[4,3,6]]`, `capacity = 6`. Lập mảng hiệu (chỉ các vị trí khác `0`) và tải tại từng vị trí; kết luận `True/False`. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện mảng hiệu:

In [31]:
def car_pool_todo(trips, capacity):
    diff = [0] * 1002
    for p, a, b in trips:
        pass            # TODO 1: diff[a] += p; diff[b] -= p
    cur = 0
    for d in diff:
        pass            # TODO 2: cur += d; nếu cur > capacity thì trả False
    return True


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(car_pool_todo(t, c) is w for t, c, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **quét sự kiện đã sắp xếp**:

In [32]:
class MySolution:
    def carPooling(self, trips, capacity):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().carPooling(t, c) is w for t, c, w in examples)

### Bài 5 (mini challenge): số chuyến tối đa có thể nhận theo thứ tự cho trước

Cho `trips` và `capacity`, xét các chuyến **theo thứ tự đã cho**; chuyến nào **nhận thêm vẫn thoả** mọi ràng buộc (cùng các chuyến đã nhận trước) thì nhận, ngược lại bỏ. Viết `accepted_count(trips, capacity)` trả về số chuyến được nhận. Ví dụ `accepted_count([[2,1,5],[3,3,7],[1,4,6]], 4) == 2` (nhận chuyến 1 và 3, bỏ chuyến 2).

In [33]:
def accepted_count(trips, capacity):
    pass            # TODO


# Khi làm xong:
# assert accepted_count([[2, 1, 5], [3, 3, 7], [1, 4, 6]], 4) == 2
# assert accepted_count([[2, 1, 5], [3, 3, 7]], 5) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tải tại từng km, nhớ đoạn nửa mở.
- Mức 2: `[[1,0,5],[1,5,10]]`: tại km `5` khách đầu đã xuống; `[[3,0,10],[4,3,6],[2,5,8]]`: đỉnh `3+4+2 = 9` tại km `5`.
- Mức 3: `True False False True`.

**Bài 2**

- Mức 1: `diff[0] += 2`, `diff[4] -= 2`, `diff[1] += 3`, `diff[3] -= 3`, `diff[3] += 4`, `diff[6] -= 4`.
- Mức 2: tải tích luỹ: km0: `2`; km1: `5`; km2: `5`; km3: `diff[3] = -3 + 4 = +1` nên `6`; km4: `diff[4] = -2` nên `4`; km5: `4`; km6: `diff[6] = -4` nên `0`.
- Mức 3: tải lớn nhất `6 ≤ 6` ⇒ `True`.

**Bài 3**

- Mức 1: `diff[a] += p`, `diff[b] -= p`.
- Mức 2: `cur += d`.
- Mức 3: `if cur > capacity: return False`.

**Bài 4**

- Mức 1: tạo `events = [(a, +p), (b, -p)]` cho mỗi chuyến.
- Mức 2: `events.sort()` (cùng vị trí `-p` đứng trước `+p`).
- Mức 3: cộng dồn, trả `False` khi `cur > capacity`.

**Bài 5**

- Mức 1: giữ danh sách chuyến đã nhận.
- Mức 2: thử thêm chuyến mới và kiểm tra `max load ≤ capacity` (dùng `Solution().carPooling`).
- Mức 3: nếu hợp lệ thì thêm và tăng đếm.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().carPooling([[1, 0, 5], [1, 5, 10]], 1), Solution().carPooling([[2, 1, 3], [2, 2, 4]], 3), Solution().carPooling([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 8), Solution().carPooling([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 9))
print(vals)
assert vals == (True, False, False, True)
assert vals == (car_pool_brute([[1, 0, 5], [1, 5, 10]], 1), car_pool_brute([[2, 1, 3], [2, 2, 4]], 3), car_pool_brute([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 8), car_pool_brute([[3, 0, 10], [4, 3, 6], [2, 5, 8]], 9))
print("Bài 1 OK")

(True, False, False, True)
Bài 1 OK


### Bài 2

In [35]:
t2 = [[2, 0, 4], [3, 1, 3], [4, 3, 6]]
assert trace_diff(t2, 6) is True and trace_diff(t2, 5) is False
assert [load_at(t2, x) for x in range(7)] == [2, 5, 5, 6, 4, 4, 0]
print("Bài 2 OK: tải [2, 5, 5, 6, 4, 4, 0], lớn nhất 6")

trips = [[2, 0, 4], [3, 1, 3], [4, 3, 6]], capacity = 6, diff = {0: 2, 1: 3, 3: 1, 4: -2, 6: -4}
  km 0: diff +2 -> tải 2 (ok)
  km 1: diff +3 -> tải 5 (ok)
  km 3: diff +1 -> tải 6 (ok)
  km 4: diff -2 -> tải 4 (ok)
  km 6: diff -4 -> tải 0 (ok)
trips = [[2, 0, 4], [3, 1, 3], [4, 3, 6]], capacity = 5, diff = {0: 2, 1: 3, 3: 1, 4: -2, 6: -4}
  km 0: diff +2 -> tải 2 (ok)
  km 1: diff +3 -> tải 5 (ok)
  km 3: diff +1 -> tải 6 (vượt 5: trả False)
Bài 2 OK: tải [2, 5, 5, 6, 4, 4, 0], lớn nhất 6


### Bài 3

In [36]:
def car_pool_answer(trips, capacity):
    diff = [0] * 1002
    for p, a, b in trips:
        diff[a] += p
        diff[b] -= p
    cur = 0
    for d in diff:
        cur += d
        if cur > capacity:
            return False
    return True


assert all(car_pool_answer(t, c) is w for t, c, w in examples)
for t, c in pool:
    assert car_pool_answer(t, c) is (max_load(t) <= c)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def carPooling(self, trips, capacity):
        events = sorted([(a, p) for p, a, b in trips] + [(b, -p) for p, a, b in trips])
        cur = 0
        for _, d in events:
            cur += d
            if cur > capacity:
                return False
        return True


assert all(MySolutionAnswer().carPooling(t, c) is w for t, c, w in examples)
for t, c in pool:
    assert MySolutionAnswer().carPooling(t, c) is (max_load(t) <= c)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def accepted_count(trips, capacity):
    accepted = []
    for trip in trips:
        if Solution().carPooling(accepted + [trip], capacity):
            accepted.append(trip)
    return len(accepted)


assert accepted_count([[2, 1, 5], [3, 3, 7], [1, 4, 6]], 4) == 2
assert accepted_count([[2, 1, 5], [3, 3, 7]], 5) == 2
rng = random.Random(19)
for _ in range(100):
    trips = rand_trips(rng, rng.randint(1, 8), maxpos=12, maxp=4)
    cap = rng.randint(1, 8)
    acc = []
    for trip in trips:
        if max_load(acc + [trip]) <= cap:
            acc.append(trip)
    assert accepted_count(trips, cap) == len(acc) and max_load(acc) <= cap
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đoạn nửa mở có trọng số, tải lớn nhất ≤ giới hạn?):**

```python
class Solution:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        diff = [0] * 1002
        for p, a, b in trips:
            diff[a] += p
            diff[b] -= p
        cur = 0
        for d in diff:
            cur += d
            if cur > capacity:
                return False
        return True
```

**Checklist trước khi submit:**

- [ ] Đoạn nửa mở `[from, to)`: `+p` tại `from`, `-p` tại `to` (không phải `to + 1`)?
- [ ] Mảng hiệu đủ lớn cho `to = 1000` (`≥ 1001` phần tử)?
- [ ] Kiểm tra **tổng tích luỹ** `cur`, không phải từng `diff`?
- [ ] So sánh `cur > capacity` (bằng thì hợp lệ)?
- [ ] Nếu quét sự kiện: sắp xếp `(vị trí, delta)` để trả trước đón khi hoà?
- [ ] Đã thử `to_i = from_j`, `capacity` đúng bằng tải lớn nhất, chuyến `[0, 1000]`?
- [ ] Đúng tên `class Solution`, `carPooling(self, trips, capacity)`, không in debug.